# Jow @ EXIST 2026 — Task 2 (Memes): Deteção de Sexismo Multimodal

Este notebook é uma **reescrita legível** de `Thesis_main.ipynb`, o notebook original de
treino do modelo de memes. O comportamento é o mesmo (mesmas contas, mesmos resultados)
— o que muda é a organização: cada bloco de código tem uma célula de texto explicando
**o que ele faz e por quê**, escrita para quem não trabalha com Machine Learning no dia a dia.

**O que o modelo faz:** recebe um meme (imagem + texto) e prevê se ele é sexista, e se sim,
de que forma. Como pessoas diferentes discordam ao rotular memes como sexistas ou não, o
modelo não aprende "sim ou não" — aprende a **distribuição de opiniões** dos anotadores
(ex.: "4 de 6 anotadores disseram YES, 2 disseram NO" → modelo aprende [0.67, 0.33],
não só "YES"). Essa abordagem chama-se **Learning with Disagreement (LeWiDi)**.

**As 3 perguntas que o modelo responde para cada meme (subtasks):**
- **Task A** — É sexista? (YES / NO)
- **Task B** — Se for, qual a intenção? (NONE / JUDGEMENTAL — julga sem atacar / DIRECT — ataque direto)
- **Task C** — Quais categorias de sexismo aparecem? (pode ser mais de uma: desigualdade
  ideológica, misoginia não-sexual, violência sexual, objetificação, estereótipos de dominância)


## Glossário de conceitos usados neste notebook

| Termo | O que significa aqui |
|---|---|
| **Token / tokenizar** | Cortar o texto em pedacinhos (palavras ou sub-palavras) que o modelo consegue processar como números. |
| **CLS token** | Um token especial adicionado no início de todo texto/imagem processado por um Transformer. Depois de passar pelo modelo, o vetor desse token resume "o significado geral" daquele texto ou imagem inteira — é o que usamos como representação da entrada. |
| **Embedding / vetor** | Uma lista de números que representa algo (uma palavra, uma imagem, uma pessoa) de um jeito que o modelo consegue comparar matematicamente. |
| **Soft label (label suave)** | Em vez de um rótulo único ("YES"), uma distribuição de probabilidade que reflete o desacordo entre anotadores, ex. `[0.67, 0.33]` para `[YES, NO]`. |
| **Hard label (label duro)** | O rótulo único que se obtém "arredondando" o soft label (ex.: pega a opção com maior probabilidade, ou aplica um corte/threshold). |
| **Gate (portão)** | Um número entre 0 e 1 (calculado pelo próprio modelo via `sigmoid`) que decide "quanto" de um sinal deixar passar. Funciona como um botão de volume que o modelo aprende a girar sozinho. |
| **LayerNorm** | Uma operação que reescala os números de um vetor para que fiquem numa faixa estável — evita que alguns números "explodam" e dominem o treino. |
| **Dropout** | Durante o treino, desliga aleatoriamente uma fração dos neurônios a cada passo — força o modelo a não depender demais de um único caminho, reduzindo overfitting. |
| **BCEWithLogitsLoss** | Função de erro usada quando a resposta é binária (sim/não) ou múltiplas respostas independentes (Task C, onde mais de uma categoria pode estar certa ao mesmo tempo). |
| **Softmax + cross-entropy** | Função de erro usada quando as opções são mutuamente exclusivas (Task B: só uma intenção por vez). |
| **Fine-tuning parcial / congelar camadas** | "Congelar" uma parte do modelo pré-treinado significa não deixá-la mudar durante o treino (economiza memória e evita destruir o que ela já sabe); só as últimas camadas ficam "descongeladas" (treináveis). |
| **ICM-Norm (soft / hard)** | Métrica oficial da competição EXIST, calculada pela biblioteca PyEvALL. Mede o quão perto a previsão do modelo está do gabarito — a versão *soft* compara distribuições inteiras, a versão *hard* compara rótulos únicos. |
| **Threshold (limiar)** | Valor de corte usado para transformar uma probabilidade em decisão binária, ex.: "se P(categoria) ≥ 0.35, marca essa categoria como presente". |

**Nota sobre versões:** este notebook roda na VM de treino do projeto — `transformers==5.6.0`,
`torch==2.6.0+cu124` (ver `CLAUDE.md` do repositório para a lista completa de versões).


## 1. Instalação de dependências

In [1]:
%pip install transformers
%pip install accelerate
%pip install emoji wordsegment
%pip install huggingface_hub
%pip install torch --index-url https://download.pytorch.org/whl/cu124
%pip install pandas scikit-learn
%pip install ftfy regex tqdm
%pip install PyEvALL

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [ ]:
import os
import re
import json
import csv
import tempfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import emoji
from wordsegment import load as load_wordsegment_dictionary, segment

from sklearn.model_selection import train_test_split

from transformers import (
    XLMRobertaTokenizer, XLMRobertaModel,
    CLIPProcessor, CLIPVisionModel,
    TrainingArguments, Trainer, TrainerCallback,
)

from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils

os.chdir("/home/a67606/Thesis/EXIST_MEMES")
# wordsegment precisa do dicionário de frequência de palavras carregado antes de segment() funcionar
load_wordsegment_dictionary()

## 2.1 Reprodutibilidade — seed global e kernels determinísticos

Sem esta seção, duas runs "idênticas" produzem pesos finais diferentes, por **três** motivos independentes:

1. **Init das camadas novas sem seed.** O `MultiHeadModel` é instanciado na seção 11.1, e as camadas criadas do zero (`text_proj`, `vision_proj`, `demo_proj`, `demo_gate`, `fusion_norm`, `head_a/b/c`) são inicializadas com o gerador global do PyTorch — que nunca era semeado. O `Trainer` até chama `set_seed(42)` internamente, mas só no próprio construtor (seção 15), **depois** de o modelo já ter nascido com pesos aleatórios diferentes a cada run. Ponto de partida diferente ⇒ trajetória de treino diferente ⇒ modelo final diferente.
2. **Kernels CUDA não-determinísticos.** Mesmo com todas as seeds fixas, várias operações de GPU (backward de embeddings, somas atômicas, algoritmos escolhidos por benchmark do cuDNN) somam floats em ordem que varia de execução para execução. O erro de arredondamento é minúsculo num passo isolado, mas realimenta o gradiente e se amplifica ao longo de 30 épocas.
3. **Seed do `Trainer` implícita.** `seed`/`data_seed` não eram passados no `TrainingArguments` — funcionava por coincidência (o default é 42). Agora ficam explícitos, porque é essa seed que controla o embaralhamento dos batches e o dropout durante o treino.

**O que a célula abaixo garante:** rodando o notebook de cima a baixo (Run All), na mesma máquina e com as mesmas versões de bibliotecas, duas runs produzem os mesmos pesos bit a bit — verificável comparando o `model_fingerprint` impresso na seção 11.1 (pesos iniciais) e na seção 16 (pesos finais) entre as runs.

**Custos e limites:**
- Kernels determinísticos custam tipicamente ~10% de velocidade de treino.
- Se alguma operação não tiver implementação determinística, o PyTorch levanta `RuntimeError` dizendo qual operador é — preferível a ser silenciosamente irreprodutível. Se acontecer, reportar o erro antes de relaxar o modo estrito.
- Reprodutibilidade vale **na mesma GPU/driver/versões** (a VM). Trocar de hardware ou de versão do PyTorch pode mudar os números — determinismo ≠ portabilidade.
- Reexecutar células fora de ordem consome os geradores em pontos diferentes; para comparar runs, sempre Run All.


In [ ]:
import hashlib
import random

SEED = 42

# O cuBLAS lê esta variável quando cria o handle (na primeira operação em GPU);
# precisa estar definida antes de qualquer uso de CUDA para ter efeito.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_full_determinism(seed=SEED):
    """Fixa todos os geradores de números aleatórios e força kernels determinísticos.

    Chamar imediatamente antes de cada passo cuja aleatoriedade precisa ser
    reproduzível (criação do modelo, treino). Chamar de novo "re-zera" os
    geradores para o mesmo ponto, independentemente do que rodou antes.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)           # gerador da CPU (init de pesos, dropout em CPU)
    torch.cuda.manual_seed_all(seed)  # geradores de todas as GPUs (dropout em GPU)

    # Sem isto, o cuDNN escolhe algoritmos por benchmark de tempo de execução
    # (varia de run para run) e usa kernels com somas atômicas em ordem
    # arbitrária — diferenças de arredondamento que se amplificam no treino.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Exige implementação determinística de TODO operador; se algum não tiver,
    # o PyTorch levanta RuntimeError apontando o operador culpado.
    torch.use_deterministic_algorithms(True)


def model_fingerprint(model):
    """Hash SHA-256 (16 hex) de todos os pesos do modelo.

    Dois modelos com o mesmo fingerprint têm pesos bit a bit idênticos —
    comparar o valor impresso entre runs verifica a reprodutibilidade.
    """
    digest = hashlib.sha256()
    for name, param in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(param.detach().cpu().numpy().tobytes())
    return digest.hexdigest()[:16]


set_full_determinism(SEED)


## 3. Carregar os dados

O dataset (`EXIST2026_training.json`) vem como um dicionário `{id_do_meme: {...campos...}}`.
`pd.read_json(...).transpose()` transforma isso numa tabela (DataFrame) com uma linha por meme.


In [3]:
raw_data = pd.read_json('./Dataset/memes/training/EXIST2026_training.json')
raw_data = raw_data.transpose()
raw_data.head()

FileNotFoundError: File ./Dataset/memes/training/EXIST2026_training.json does not exist

### 3.1 Checagem rápida: quantos memes têm sinais biométricos?

Alguns memes têm dados extras coletados dos anotadores enquanto olhavam a imagem (rastreamento
ocular = ET, frequência cardíaca = HR, eletroencefalograma = EEG). **Este notebook (Task 2 —
memes) não usa esses sinais** — eles pertencem ao pipeline da Task 3 (vídeos). A checagem
abaixo é só um raio-x do dataset, para confirmar o que está disponível.


In [ ]:
def has_biometric_modality(row, modality_name):
    """Verifica se um meme tem dados biométricos (ET/HR/EEG) de pelo menos 1 anotador."""
    sensorial_data = row.get('sensorial')
    if sensorial_data is None or not isinstance(sensorial_data, dict):
        return False
    users_with_data = sensorial_data.get('modalities', {}).get(modality_name, {}).get('by_user', {})
    return len(users_with_data) > 0

print(f"Total de memes: {len(raw_data)}")
print(f"Com algum dado biométrico: {raw_data.apply(lambda r: r.get('sensorial') is not None, axis=1).sum()}")
for modality in ['ET', 'HR', 'EEG']:
    count = raw_data.apply(lambda r, modality=modality: has_biometric_modality(r, modality), axis=1).sum()
    print(f"Com {modality}: {count}")

## 4. Remover colunas que não usamos

`id_EXIST`, `number_annotators`, `annotators`, `lang`, `split` são metadados administrativos
do dataset — não entram no modelo.


In [ ]:
dataset = raw_data.copy()
dataset.drop(['id_EXIST', 'number_annotators', 'annotators', 'lang', 'split'], axis=1, inplace=True)
dataset.describe()

In [ ]:
for i in range(3):
    path = os.path.join('./Dataset/memes/training/', dataset['path_memes'].iloc[i])
    exists = os.path.exists(path)
    print(f"{path} -> {'existe' if exists else 'FALTANDO'}")

## 5. Definições de labels (rótulos)

Cada subtask tem sua própria lista fixa de rótulos possíveis. `THRESHOLD_GRID` é a lista de
valores de corte que testamos mais adiante para decidir "esse limiar de probabilidade conta
como presente ou não" (Task C). As hierarquias (`TASK_2_2_HIERARCHY`, `TASK_2_3_HIERARCHY`)
dizem à métrica oficial (PyEvALL) que, por exemplo, errar "DIRECT" por "JUDGEMENTAL" é um erro
menor do que errar "sexista" por "não-sexista" — ambos são sub-categorias de "YES".

⚠️ Estas definições existem **uma única vez** neste notebook. No notebook original elas
apareciam duplicadas em duas células — inofensivo porque os valores batiam, mas é um padrão
arriscado (se alguém editasse só uma cópia, os índices dessincronizariam silenciosamente).


In [ ]:
LABELS_2_1 = ['YES', 'NO']
LABELS_2_2 = ['NONE', 'JUDGEMENTAL', 'DIRECT']
LABELS_2_3 = ['NONE', 'IDEOLOGICAL-INEQUALITY', 'MISOGYNY-NON-SEXUAL-VIOLENCE',
              'SEXUAL-VIOLENCE', 'OBJECTIFICATION', 'STEREOTYPING-DOMINANCE']

# Valores de corte testados na busca de threshold da Task C (seção 11)
THRESHOLD_GRID = np.arange(0.05, 0.65, 0.05).tolist()   # [0.05, 0.10, ..., 0.60]

TASK_2_2_HIERARCHY = {
    "YES": ["DIRECT", "JUDGEMENTAL"],
    "NO": []
}

TASK_2_3_HIERARCHY = {
    "YES": [
        "IDEOLOGICAL-INEQUALITY",
        "STEREOTYPING-DOMINANCE",
        "OBJECTIFICATION",
        "SEXUAL-VIOLENCE",
        "MISOGYNY-NON-SEXUAL-VIOLENCE"
    ],
    "NO": []
}

## 6. Limpeza de texto

Antes de mandar o texto do meme para o modelo de linguagem, aplicamos uma sequência de limpezas.
Cada função abaixo faz **uma coisa só** — dá pra testar cada passo isoladamente.

✅ **Corrigido nesta revisão (2026-08-01):** `normalize_hashtags` tenta segmentar hashtags
sem underscore/CamelCase usando a biblioteca `wordsegment` (ex.: `#mygreatidea` -> `my great
idea`). Essa biblioteca precisa de uma chamada explícita a `wordsegment.load()` para carregar
seu dicionário de frequência de palavras — e o notebook original (`Thesis_main.ipynb`)
**nunca chamava `load()`**. Sem isso, `segment()` lançava `ValueError` (dicionário vazio) em
toda chamada, e o `except ValueError` capturava o erro e devolvia a hashtag sem segmentar
(`words = [tag]`). Ou seja: **esse ramo de segmentação por dicionário nunca funcionou no
sistema submetido** — toda hashtag sem underscore/CamelCase virava uma palavra só, grudada.
Este notebook agora chama `load()` na célula de imports (seção 2), então a segmentação
funciona de verdade. **Isso muda o texto de entrada do modelo** em 10 das 3.984 amostras do
dataset de treino (as que têm hashtag segmentável) — o modelo treinado a partir daqui não é
mais idêntico ao sistema submetido nesse detalhe.


In [ ]:
def normalize_hashtags(text):
    """Transforma hashtags em palavras legíveis.
    Ex.: '#MyGreatIdea' -> 'my great idea', '#happy_day' -> 'happy day'.
    Tenta 3 estratégias, na ordem: separar por underscore, separar por
    CamelCase, ou usar um dicionário de segmentação de palavras (wordsegment)
    como último recurso."""
    def split_one_hashtag(match):
        tag_without_hash = match.group()[1:]
        if "_" in tag_without_hash:
            words = tag_without_hash.split("_")
        elif re.search(r'[A-Z]', tag_without_hash):
            words = re.sub(r'(?<!^)(?=[A-Z])', ' ', tag_without_hash).split()
        else:
            try:
                words = segment(tag_without_hash)
            except ValueError:
                words = [tag_without_hash]  # não conseguiu segmentar, mantém como está
        return " ".join(words)
    return re.sub(r'#\w+', split_one_hashtag, text)

In [ ]:
def convert_emojis_to_text(text):
    """Troca emoji por sua descrição textual, ex.: 😂 -> ':face_with_tears_of_joy:'.
    O tokenizador entende palavras muito melhor do que glifos de emoji."""
    return emoji.demojize(text)

In [ ]:
def replace_urls(text):
    """Qualquer link vira o placeholder <URL> — o conteúdo do link em si não importa
    para detectar sexismo, só atrapalharia o tokenizador."""
    return re.sub(r'http\S+|www\S+', '<URL>', text)

In [ ]:
def replace_user_mentions(text):
    """@fulano -> <USER_MENTION>. Anonimiza e generaliza menções a usuários."""
    return re.sub(r'@\w+', '<USER_MENTION>', text)

In [ ]:
def replace_profanity(text):
    """Palavrões censurados com símbolos (ex.: 'f*ck', 'sh!t') viram <PROFANITY>.
    Detecta: qualquer 'palavra' que contenha um dos símbolos * ! @ # $ % ^ &."""
    profanity_pattern = r'\b[a-zA-Z]*[*!@#$%^&]+[a-zA-Z]*\b'
    return re.sub(profanity_pattern, "<PROFANITY>", text)

In [ ]:
def limit_token_repetition(text, max_repeats=2):
    """Corta repetições consecutivas da mesma palavra em no máximo `max_repeats`.
    Ex.: 'não não não não não' -> 'não não'. Mantém a ênfase sem deixar o texto
    virar uma parede da mesma palavra repetida, que confundiria o tokenizador."""
    words = text.split()
    kept_words = []
    previous_word = None
    repeat_count = 0
    for word in words:
        if word == previous_word:
            repeat_count += 1
        else:
            previous_word = word
            repeat_count = 1
        if repeat_count <= max_repeats:
            kept_words.append(word)
    return " ".join(kept_words)

In [ ]:
def strip_punctuation_and_extra_spaces(text):
    """Remove qualquer caractere que não seja letra/número/underscore/espaço/< >,
    depois colapsa espaços múltiplos em um só. Os placeholders <URL>, <USER_MENTION>,
    <PROFANITY> sobrevivem porque < e > estão na lista de exceções."""
    text = re.sub(r'[^\w\s<>]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
def pre_process_text(text):
    """Pipeline completo de limpeza de texto, aplicado em ordem:
    1. hashtags -> palavras legíveis
    2. tudo em minúsculas
    3. URLs -> <URL>
    4. emoji -> descrição textual
    5. @menções -> <USER_MENTION>
    6. palavrões censurados -> <PROFANITY>
    7. limita repetição de palavras consecutivas
    8. remove pontuação sobrando e espaços extras
    """
    text = normalize_hashtags(text)
    text = text.lower()
    text = replace_urls(text)
    text = convert_emojis_to_text(text)
    text = replace_user_mentions(text)
    text = replace_profanity(text)
    text = limit_token_repetition(text)
    text = strip_punctuation_and_extra_spaces(text)
    return text

### 6.1 Exemplo de ponta a ponta

Para deixar claro o que o pipeline faz na prática, um exemplo real do dataset, antes e depois:


In [ ]:
example_raw = dataset['text'].iloc[0]
example_clean = pre_process_text(example_raw)
print("ANTES:")
print(example_raw)
print()
print("DEPOIS:")
print(example_clean)

## 7. Rótulos suaves (soft labels) e remapeamento

Cada meme tem entre 3 e 6 anotadores dando seu voto. Duas coisas precisam de tratamento
especial:

1. **Voto `-`** (usado pelo esquema de anotação para "não é sexista") deve virar `NONE`,
   **nunca ser descartado** — descartar infla artificialmente a proporção de
   `DIRECT`/`JUDGEMENTAL`, porque o denominador (total de votos) diminuiria sem motivo.
2. **Voto `UNKNOWN`** (o anotador não conseguiu decidir) é descartado dos targets de treino
   — não representa uma opinião real sobre o conteúdo.

Depois do remapeamento, contamos os votos e transformamos em uma distribuição de
probabilidade — isso é o "soft label" mencionado no glossário.


In [ ]:
def remap_labels_flat(annotator_labels):
    """Para tasks onde cada anotador dá UM rótulo (Task A, Task B).
    '-' (não-sexista) -> 'NONE'. 'UNKNOWN' (indeciso) é descartado."""
    return [
        "NONE" if label == "-" else label
        for label in annotator_labels
        if label != "UNKNOWN"
    ]

def remap_labels_nested(per_annotator_label_lists):
    """Mesma regra acima, mas para tasks onde cada anotador dá VÁRIOS rótulos (Task C).
    Um anotador cuja lista fica vazia depois do remapeamento é removido inteiramente
    (ele não deu nenhuma opinião válida)."""
    cleaned_lists = []
    for this_annotators_labels in per_annotator_label_lists:
        cleaned = [
            "NONE" if label == "-" else label
            for label in this_annotators_labels
            if label != "UNKNOWN"
        ]
        if cleaned:
            cleaned_lists.append(cleaned)
    return cleaned_lists

In [ ]:
def get_soft_distribution(annotator_labels, all_possible_labels):
    """Conta os votos de uma lista FLAT (Task A, Task B) e devolve a fração de cada rótulo.
    Ex.: ['YES','YES','NO'] com labels ['YES','NO'] -> [0.667, 0.333].
    Devolve None se não houve nenhum voto (a amostra deve ser descartada)."""
    if not annotator_labels:
        return None
    vote_counts = dict.fromkeys(all_possible_labels, 0)
    for vote in annotator_labels:
        if vote in vote_counts:
            vote_counts[vote] += 1
    num_votes = len(annotator_labels)
    return [vote_counts[label] / num_votes for label in all_possible_labels]

def annotations_matrix_to_distribution(per_annotator_label_lists, all_possible_labels):
    """Mesma ideia, mas para a Task C, onde cada anotador pode escolher MAIS DE UM
    rótulo. Conta quantos ANOTADORES (não quantas escolhas) marcaram cada categoria,
    dividido pelo número de anotadores."""
    if not per_annotator_label_lists:
        return None
    label_counts = dict.fromkeys(all_possible_labels, 0)
    num_annotators = len(per_annotator_label_lists)
    for this_annotators_labels in per_annotator_label_lists:
        for label in this_annotators_labels:
            if label in label_counts:
                label_counts[label] += 1
    return [label_counts[label] / num_annotators for label in all_possible_labels]

## 8. Vetor demográfico dos anotadores

A ideia (inspirada na abordagem "GrootWatch"): se soubermos o perfil demográfico de quem
anotou cada meme (etnia, escolaridade, país), o modelo pode aprender que certos grupos
tendem a discordar mais em certos tipos de conteúdo — e usar isso como sinal extra.

**Como o vetor é montado, passo a passo:**
1. Para cada uma das 3 características (etnia / escolaridade / país), contamos quantos dos
   ~6 anotadores daquele meme caem em cada categoria possível.
2. Concatenamos as 3 listas de contagem numa única lista de 66 números (7 etnias + 6
   escolaridades + 53 países).
3. Dividimos a lista inteira pela soma total — assim cada posição vira "fração dos votos",
   não uma contagem bruta (isso é importante: normalizar dividindo pela soma preserva o
   sinal; testamos normalizar por norma L2 e o sinal demográfico foi destruído).

✅ **Corrigido nesta revisão (2026-08-01):** os dados brutos usam apóstrofo curvo (`’`,
ex. `"Bachelor’s degree"`), mas o vocabulário abaixo usava apóstrofo reto (`'`). Isso fazia
com que **64,6% dos votos de escolaridade nunca batessem com o vocabulário** e fossem
descartados silenciosamente do vetor (a fatia de escolaridade ficava quase sempre zerada).
`_normalize_apostrophes` (na função abaixo) trata os dois caracteres como equivalentes na
hora da comparação — o vocabulário e a ordem das 66 posições continuam exatamente os mesmos,
só a lógica de correspondência mudou. Efeito: **3.957 das 3.984 amostras têm o vetor
demográfico alterado** em relação ao sistema submetido (a normalização por soma agora usa o
denominador correto).


In [ ]:
DEMO_ETHNICITY_VALS = sorted(['Asian', 'Black or African American', 'Hispano or Latino',
                               'Middle Eastern', 'Multiracial', 'White or Caucasian', 'other'])
DEMO_STUDY_VALS = sorted(["Bachelor's degree", 'Doctorate',
                           'High school degree or equivalent',
                           'Less than high school diploma', "Master's degree", 'other'])
DEMO_COUNTRY_VALS = sorted(['Afghanistan', 'Argentina', 'Australia', 'Austria', 'Azerbaijan',
                             'Bangladesh', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile',
                             'China', 'Colombia', 'Croatia', 'Cuba', 'Czech Republic',
                             'Dominican Republic', 'Ecuador', 'Estonia', 'Finland', 'France',
                             'Germany', 'Greece', 'Hungary', 'India', 'Ireland', 'Israel',
                             'Italy', 'Latvia', 'Mexico', 'Morocco', 'Netherlands',
                             'New Zealand', 'Nicaragua', 'Nigeria', 'Peru', 'Poland',
                             'Portugal', 'Romania', 'Russian Federation', 'Serbia', 'Slovenia',
                             'South Africa', 'Spain', 'Sweden', 'Turkey', 'United Arab Emirates',
                             'United Kingdom', 'United States', 'Uruguay', 'Venezuela',
                             'Viet Nam', 'Zimbabwe'])

# (nome da coluna no dataset, vocabulário fixo daquela característica)
DEMO_FEATURE_GROUPS = [
    ('ethnicities_annotators',  DEMO_ETHNICITY_VALS),
    ('study_levels_annotators', DEMO_STUDY_VALS),
    ('countries_annotators',    DEMO_COUNTRY_VALS),
]

DEMO_VEC_DIM = len(DEMO_ETHNICITY_VALS) + len(DEMO_STUDY_VALS) + len(DEMO_COUNTRY_VALS)  # 7+6+53 = 66
print(f"Dimensão do vetor demográfico: {DEMO_VEC_DIM}")

In [ ]:
def _normalize_apostrophes(text):
    """Trata apóstrofo curvo (’, U+2019) e reto (', U+0027) como o mesmo caractere.
    Os dados brutos usam ’ em "Bachelor’s degree"/"Master’s degree"; o vocabulário fixo
    abaixo usa '. Sem essa normalização, 64,6% dos votos de escolaridade nunca batiam
    com o vocabulário e eram descartados silenciosamente (achado desta revisão de código,
    corrigido aqui -- ver seção 8)."""
    return text.replace("’", "'")


def _count_votes_per_category(annotator_values, vocabulary):
    """Para UM grupo de característica (ex.: escolaridade): conta quantos anotadores
    caem em cada categoria do vocabulário. Devolve um vetor de contagens, na mesma
    ordem do vocabulário."""
    category_to_index = {
        _normalize_apostrophes(category): i for i, category in enumerate(vocabulary)
    }
    counts = np.zeros(len(vocabulary), dtype=np.float32)
    if not isinstance(annotator_values, list):
        return counts
    for value in annotator_values:
        if value is None:
            continue
        idx = category_to_index.get(_normalize_apostrophes(str(value).strip()), None)
        if idx is not None:
            counts[idx] += 1.0
    return counts


def compute_annotator_vector(row, feature_groups):
    """Monta o vetor demográfico de 66 posições para um meme.

    Para cada grupo de característica: conta quantos anotadores caem em cada
    categoria daquele vocabulário (_count_votes_per_category), concatena as 3
    contagens, e divide o vetor inteiro pela soma total (normalização por soma,
    não por norma L2).
    """
    category_count_vectors = [
        _count_votes_per_category(row.get(column_name, None), vocabulary)
        for column_name, vocabulary in feature_groups
    ]

    full_vector = np.concatenate(category_count_vectors)  # [66]
    total_votes = full_vector.sum()
    if total_votes > 1e-9:
        full_vector = full_vector / total_votes
    return full_vector

## 9. Aplicar tudo ao dataset

Agora rodamos o pré-processamento de texto, o remapeamento + soft labels de cada task, e o
vetor demográfico, em todas as linhas do dataset. Por fim, removemos as poucas amostras sem
nenhum voto válido na Task A (a task principal — sem ela, não há o que treinar).


In [ ]:
dataset['text_preprocessed'] = dataset['text'].apply(pre_process_text)

dataset['labels_task2_1_soft'] = dataset['labels_task2_1'].apply(
    lambda votes: get_soft_distribution(votes, LABELS_2_1)
)

dataset['labels_task2_2'] = dataset['labels_task2_2'].apply(remap_labels_flat)
dataset['labels_task2_2_soft'] = dataset['labels_task2_2'].apply(
    lambda votes: get_soft_distribution(votes, LABELS_2_2)
)

dataset['labels_task2_3'] = dataset['labels_task2_3'].apply(remap_labels_nested)
dataset['labels_task2_3_soft'] = dataset['labels_task2_3'].apply(
    lambda votes: annotations_matrix_to_distribution(votes, LABELS_2_3)
)

dataset['demo_vec'] = dataset.apply(
    lambda row: compute_annotator_vector(row, DEMO_FEATURE_GROUPS), axis=1
)

print(f"Antes de filtrar: {len(dataset)} amostras")
dataset = dataset[dataset['labels_task2_1_soft'].notna()].reset_index(drop=True)
print(f"Depois de filtrar (Task A precisa de ao menos 1 voto válido): {len(dataset)} amostras")
dataset.head()

## 10. O "Dataset" do PyTorch

Um `Dataset` do PyTorch é só um objeto que sabe responder duas perguntas: "quantas amostras
existem?" (`__len__`) e "me dá a amostra número N, já pronta pro modelo" (`__getitem__`).
Aqui, "pronta pro modelo" significa: texto tokenizado, imagem processada pelo CLIP, vetor
demográfico, e os 3 soft labels — tudo como tensores do PyTorch.

**Casos especiais tratados abaixo:**
- **Imagem ausente/corrompida** → uma imagem preta 224×224 é usada no lugar (não quebra o
  treino, mas o sinal daquela amostra fica "vazio" para o CLIP).
- **Meme sem voto válido na Task B ou C** → em vez de um soft label de verdade, colocamos um
  vetor "sentinela" preenchido com `-1.0`. Isso é um sinalizador especial (não é uma
  distribuição de probabilidade real!) que o modelo detecta mais adiante e usa para **ignorar
  essa amostra** no cálculo daquela loss específica, sem descartar a amostra inteira (as
  outras tasks daquela amostra continuam valendo).

Renomeei `self.labels_task1/2/3` (nomes do notebook original) para `self.labels_a/b/c` —
são exatamente a Task A/B/C, e o nome antigo (`task1`/`task2`/`task3`) confundia com os
nomes de coluna do dataset (`labels_task2_1`, `labels_task2_2`...), que vêm do formato
oficial do EXIST e não podem ser renomeados.


In [ ]:
class SexismDataset(Dataset):
    def __init__(self, dataframe, tokenizer, clip_processor, max_length, image_dir):
        self.tokenizer      = tokenizer
        self.clip_processor = clip_processor
        self.max_length     = max_length
        self.image_dir      = image_dir

        self.text        = dataframe['text_preprocessed'].values
        self.image_paths = dataframe['path_memes'].values
        self.labels_a    = dataframe['labels_task2_1_soft'].values
        self.labels_b    = dataframe['labels_task2_2_soft'].values
        self.labels_c    = dataframe['labels_task2_3_soft'].values
        self.demo_vecs   = dataframe['demo_vec'].values  # [N,] de np.ndarray [66]

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        tokenized = self.tokenizer(
            self.text[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        image_path = os.path.join(self.image_dir, self.image_paths[idx])
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            print("Imagem não encontrada:", image_path)
            image = Image.new('RGB', (224, 224), color=0)  # fallback: imagem preta

        image_inputs = self.clip_processor(images=image, return_tensors='pt')

        batch = {
            'input_ids':      tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0),
            'pixel_values':   image_inputs['pixel_values'].squeeze(0),
            'demo_vec':       torch.tensor(self.demo_vecs[idx], dtype=torch.float32),  # [66]
            'labels_a':       torch.tensor(self.labels_a[idx], dtype=torch.float32),
        }

        # Task B: se não há soft label válido, usa o vetor sentinela -1.0
        # (o modelo detecta isso e ignora a amostra no cálculo da loss_b)
        label_b = self.labels_b[idx]
        if label_b is None:
            batch['labels_b'] = torch.full((len(LABELS_2_2),), -1.0, dtype=torch.float32)
        else:
            label_b = np.array(label_b, dtype=np.float32)
            if label_b[0] != -1.0:
                batch['labels_b'] = torch.tensor(label_b, dtype=torch.float32)
            else:
                batch['labels_b'] = torch.full((len(label_b),), -1.0, dtype=torch.float32)

        # Task C: mesma lógica de sentinela -1.0
        label_c = self.labels_c[idx]
        if label_c is None:
            batch['labels_c'] = torch.full((len(LABELS_2_3),), -1.0, dtype=torch.float32)
        else:
            label_c = np.array(label_c, dtype=np.float32)
            if label_c[0] != -1.0:
                batch['labels_c'] = torch.tensor(label_c, dtype=torch.float32)
            else:
                batch['labels_c'] = torch.full((len(label_c),), -1.0, dtype=torch.float32)

        return batch

## 11. A arquitetura do modelo (`MultiHeadModel`)

O modelo tem 3 "entradas" (texto, imagem, demografia dos anotadores) que se combinam num
único vetor, e 3 "cabeças" de saída (uma por subtask). Em alto nível:

```
texto  ──[XLM-RoBERTa-large]──> CLS token (1024-d) ──[projeta p/ 512-d]──┐
                                                                          │
demografia (66-d) ──[gate + projeção]────────────────> soma no texto ───┤
                                                                          ├──[concatena]──> 1024-d ──> 3 cabeças ──> 3 previsões
imagem ──[CLIP ViT-L/14]──> CLS token (1024-d) ──[projeta p/ 512-d]──────┘
```

**Por que concatenar em vez de usar cross-attention (texto "olhando" para a imagem e
vice-versa)?** Cross-attention foi testada e descartada durante o desenvolvimento: com um
dataset pequeno (~5.000 memes), ela colapsava para padrões quase uniformes (o mecanismo de
atenção não conseguia aprender nada útil e virava só ruído). A concatenação simples, mais
"burra", generalizou melhor.

**Por que a demografia entra por "soma com portão" (gate) e não por concatenação?** A
concatenação de um vetor pequeno (66-d) direto num vetor grande (1024-d) faz o sinal
demográfico ficar "diluído" — o modelo tende a ignorá-lo. Somar ao vetor de texto, com um
portão que o próprio modelo aprende a abrir ou fechar por amostra, deu resultado melhor:
`texto_final = texto + portão × demografia_projetada`, onde `portão = sigmoid(...)` é um
número entre 0 e 1 por posição do vetor — funciona como um controle de volume que o modelo
ajusta sozinho durante o treino.

**Por que só as últimas 4 camadas do CLIP ficam treináveis?** O CLIP já vem pré-treinado
com bastante conhecimento visual geral. Treinar o modelo inteiro num dataset de 5.000
imagens arriscaria "esquecer" esse conhecimento (overfitting) e custaria muito mais memória.
Só descongelamos as últimas 4 camadas, que são as mais específicas para a tarefa.

**As 3 cabeças de saída** usam funções de erro diferentes porque as perguntas são diferentes:
- **Task A** (sexista? sim/não) → `BCEWithLogitsLoss`, pergunta binária.
- **Task B** (qual intenção?) → softmax + cross-entropy contra o soft label, porque as 3
  opções (NONE/JUDGEMENTAL/DIRECT) são mutuamente exclusivas — só uma é "a resposta".
- **Task C** (quais categorias?) → `BCEWithLogitsLoss` por categoria, porque mais de uma
  categoria pode estar certa ao mesmo tempo (não são mutuamente exclusivas).

**Flags novas (`use_vision`, `use_demographics`):** o notebook original já tinha esses
comportamentos como opcionais implicitamente (`if pixel_values is not None`, `if demo_vec is
not None`). Só tornei isso explícito na assinatura do construtor — não muda nenhum
comportamento com os valores padrão (`True`, `True`), mas deixa claro, olhando só a
assinatura, que essas duas entradas são desligáveis. Isso também prepara o terreno para o
estudo de ablação (rodar o modelo só-com-texto, só-com-texto+imagem, etc.) sem precisar
mexer no código do modelo depois — **os valores padrão continuam idênticos ao notebook
original**, então isto não é uma mudança de comportamento, só uma mudança de interface.


In [ ]:
class MultiHeadModel(nn.Module):
    def __init__(self, num_labels_task_b, num_labels_task_c, demo_vec_dim=66,
                 dropout_prob=0.1, use_vision=True, use_demographics=True):
        super().__init__()
        self.use_vision = use_vision
        self.use_demographics = use_demographics

        # === Codificador de texto ===
        self.encoder = XLMRobertaModel.from_pretrained("xlm-roberta-large")
        text_dim = self.encoder.config.hidden_size  # 1024

        shared_dim = 512
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, shared_dim), nn.LayerNorm(shared_dim)
        )

        # === Codificador de imagem (CLIP), parcialmente congelado ===
        if self.use_vision:
            self.vision_encoder = CLIPVisionModel.from_pretrained(
                "openai/clip-vit-large-patch14"
            )
            vision_dim = self.vision_encoder.config.hidden_size  # 1024
            frozen_layers = 4
            for param in self.vision_encoder.parameters():
                param.requires_grad = False
            for layer in self.vision_encoder.encoder.layers[-frozen_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
            self.vision_proj = nn.Sequential(
                nn.Linear(vision_dim, shared_dim), nn.LayerNorm(shared_dim)
            )

        # === Demografia: projetada e somada (com portão) ao CLS de texto ===
        if self.use_demographics:
            self.demo_gate = nn.Linear(demo_vec_dim, shared_dim)
            self.demo_proj = nn.Sequential(
                nn.Linear(demo_vec_dim, shared_dim),
                nn.LayerNorm(shared_dim),
            )

        # === Fusão: 512 (texto+demo) + 512 (visão) = 1024 ===
        fused_dim = shared_dim * 2
        self.fusion_norm = nn.LayerNorm(fused_dim)
        self.dropout = nn.Dropout(dropout_prob)

        self.head_a = nn.Linear(fused_dim, 1)                    # Task A: 1 logit (sexista?)
        self.head_b = nn.Linear(fused_dim, num_labels_task_b)    # Task B: 3 logits (intenção)
        self.head_c = nn.Linear(fused_dim, num_labels_task_c)    # Task C: 6 logits (categorias)

        self.loss_a_fn = nn.BCEWithLogitsLoss()
        self.loss_c_fn = nn.BCEWithLogitsLoss(reduction='none')

    def _loss_task_a(self, logits_a, labels_a):
        """Task A: sexista sim/não. Devolve None se nenhuma amostra do batch tem
        voto válido (sentinela -100.0)."""
        if labels_a is None:
            return None
        # labels_a chega como [B, 2] do coletor padrão do Trainer; a coluna 1
        # é P(NO) -- ver LABELS_2_1 = ['YES', 'NO']. Depois desta linha vira [B].
        if labels_a.dim() == 2:
            labels_a = labels_a[:, 1]
        mask_a = labels_a != -100.0  # sentinela de "sem voto válido" para Task A
        if not mask_a.any():
            return None
        loss_a = self.loss_a_fn(logits_a[mask_a].view(-1), labels_a[mask_a])
        return loss_a * 2.0  # Task A tem peso 2x -- é a pergunta principal da competição

    def _loss_task_b(self, logits_b, labels_b):
        """Task B: intenção (NONE/JUDGEMENTAL/DIRECT). Cross-entropy contra soft label.
        Devolve None se nenhuma amostra do batch tem voto válido (sentinela -1.0)."""
        if labels_b is None:
            return None
        mask_b = labels_b[:, 0] != -1.0
        if not mask_b.any():
            return None
        log_probs_b = F.log_softmax(logits_b[mask_b], dim=-1)
        # cross-entropy contra soft label: -sum(p_verdadeiro * log(p_previsto))
        return -(labels_b[mask_b] * log_probs_b).sum(dim=-1).mean()

    def _loss_task_c(self, logits_c, labels_c):
        """Task C: categorias finas (multilabel). BCE por categoria.
        Devolve None se nenhuma amostra do batch tem voto válido (sentinela -1.0)."""
        if labels_c is None:
            return None
        mask_c = labels_c[:, 0] != -1.0
        if not mask_c.any():
            return None
        return self.loss_c_fn(logits_c[mask_c], labels_c[mask_c]).mean()

    def forward(self, input_ids, attention_mask,
                pixel_values=None, demo_vec=None,
                labels_a=None, labels_b=None, labels_c=None, **kwargs):

        # === Texto ===
        text_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = self.text_proj(text_out.last_hidden_state[:, 0, :])  # [B, 512]

        # === Injeta demografia no CLS de texto, com portão aprendido ===
        if self.use_demographics and demo_vec is not None:
            gate = torch.sigmoid(self.demo_gate(demo_vec))  # [B, 512], valores em (0,1)
            text_cls = text_cls + gate * self.demo_proj(demo_vec)
        # Sem demo_vec (ou use_demographics=False): text_cls fica como está

        # === Imagem ===
        if self.use_vision and pixel_values is not None:
            vision_out = self.vision_encoder(pixel_values=pixel_values)
            vision_cls = self.vision_proj(vision_out.last_hidden_state[:, 0, :])
        else:
            vision_cls = torch.zeros_like(text_cls)

        # === Fusão: concatenação simples, sem cross-attention ===
        fused = torch.cat([text_cls, vision_cls], dim=-1)  # [B, 1024]
        fused = self.fusion_norm(fused)
        fused = self.dropout(fused)

        logits_a = self.head_a(fused)
        logits_b = self.head_b(fused)
        logits_c = self.head_c(fused)

        # === Loss ===
        # Cada amostra pode não ter voto válido em Task B e/ou C (ver seção 10) —
        # cada _loss_task_* devolve None nesse caso, e é ignorada na soma abaixo,
        # sem afetar as outras tasks.
        task_losses = [
            self._loss_task_a(logits_a, labels_a),
            self._loss_task_b(logits_b, labels_b),
            self._loss_task_c(logits_c, labels_c),
        ]
        task_losses = [l for l in task_losses if l is not None]

        if task_losses:
            loss = sum(task_losses)
        else:
            # nenhuma task tinha label válido neste batch -- loss zero que ainda
            # participa do grafo computacional (evita erro do Trainer)
            loss = logits_a.sum() * 0.0

        return (loss, logits_a, logits_b, logits_c)

### 11.1 Instanciar o modelo e contar parâmetros


In [ ]:
# Re-zera os geradores imediatamente antes de criar o modelo: as camadas novas
# nascem com exatamente os mesmos pesos em qualquer run, independentemente de
# quantas células consumiram aleatoriedade antes desta.
set_full_determinism(SEED)

model = MultiHeadModel(
    num_labels_task_b=len(LABELS_2_2),
    num_labels_task_c=len(LABELS_2_3),
    demo_vec_dim=DEMO_VEC_DIM,
)

print("Fingerprint dos pesos iniciais:", model_fingerprint(model))


In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
frozen_params = total_params - trainable_params

print(f"Parâmetros totais:     {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Parâmetros congelados: {frozen_params:,}")

## 12. Separar treino e validação

Reservamos 15% dos memes para validação, nunca usados no treino em si — servem para medir
se o modelo está generalizando ou só decorando. A divisão é **estratificada** pela Task A
(garante a mesma proporção de sexista/não-sexista nos dois conjuntos) e usa uma semente fixa
(`random_state=42`) para ser reprodutível.


In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

def majority_vote_task_a(soft_label):
    """Converte o soft label da Task A no rótulo majoritário, só para estratificar o split."""
    return 'YES' if soft_label[0] >= 0.5 else 'NO'

stratify_col = dataset['labels_task2_1_soft'].apply(majority_vote_task_a)

dataset_train, dataset_val = train_test_split(
    dataset,
    test_size=0.15,
    random_state=42,
    stratify=stratify_col,
)
dataset_train = dataset_train.reset_index(drop=True)
dataset_val = dataset_val.reset_index(drop=True)

IMAGE_DIR = './Dataset/memes/training/'

train_dataset = SexismDataset(
    dataframe=dataset_train,
    tokenizer=tokenizer,
    clip_processor=clip_processor,
    max_length=128,
    image_dir=IMAGE_DIR,
)

val_dataset = SexismDataset(
    dataframe=dataset_val,
    tokenizer=tokenizer,
    clip_processor=clip_processor,
    max_length=128,
    image_dir=IMAGE_DIR,
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)

print(f"Treino: {len(train_dataset)} amostras | Validação: {len(val_dataset)} amostras")

## 13. Configuração de treino

Cada parâmetro abaixo controla um aspecto do processo de treino:

| Parâmetro | O que faz |
|---|---|
| `num_train_epochs` | Quantas vezes o modelo passa pelo dataset de treino inteiro. |
| `per_device_train/eval_batch_size` | Quantas amostras processadas juntas em cada passo. |
| `gradient_accumulation_steps` | Acumula gradiente de vários batches pequenos antes de atualizar os pesos — simula um batch maior (8×8=64) sem precisar de mais memória de GPU de uma vez. |
| `learning_rate` | O tamanho do "passo" que o modelo dá a cada atualização de pesos. |
| `warmup_steps` | Início do treino com taxa de aprendizado crescendo aos poucos (em vez de já começar no valor máximo), para não desestabilizar os pesos pré-treinados logo de cara. |
| `weight_decay` | Penaliza pesos muito grandes, ajudando a evitar overfitting. |
| `eval_strategy` / `save_strategy` | Quando avaliar/salvar um checkpoint — aqui, a cada época. |
| `load_best_model_at_end` | Ao final do treino, recarrega o checkpoint que teve o melhor resultado na validação (não necessariamente o último). |
| `metric_for_best_model` | Qual métrica decide "melhor" — aqui, o ICM-Norm soft da Task A. |
| `fp16` | Usa números de 16 bits em vez de 32 bits em partes do treino — mais rápido e usa menos memória, com perda de precisão numérica desprezível na prática. |
| `seed` / `data_seed` | Sementes dos geradores usados pelo Trainer (dropout, embaralhamento dos batches). Explícitas por reprodutibilidade — ver seção 2.1. |


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    seed=SEED,       # dropout e embaralhamento dos batches (Trainer aplica no __init__)
    data_seed=SEED,  # gerador dedicado do sampler dos dataloaders
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=0.1,
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="task_a_icmsoftnorm_soft",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    fp16=True,
    report_to="none",
    dataloader_pin_memory=True,
)

## 14. Métricas de avaliação (PyEvALL)

A competição EXIST usa a biblioteca oficial **PyEvALL** para calcular a métrica **ICM-Norm**
(Information Contrast Model, normalizada), em duas variantes:
- **soft** — compara a distribuição de probabilidade inteira prevista contra a distribuição
  real de votos dos anotadores (penaliza menos um erro "quase certo").
- **hard** — compara um rótulo único previsto contra o rótulo único do gabarito (para chegar
  num rótulo único a partir de uma distribuição, é preciso um **threshold**, ver seção 14.2).

A função `compute_metrics` roda a cada época de validação e devolve um dicionário de métricas
que o `Trainer` usa para decidir qual checkpoint é o melhor (`metric_for_best_model` acima).


### 14.1 Funções auxiliares — formatar e chamar o PyEvALL

Estas funções convertem as previsões do modelo para o formato JSON que o PyEvALL espera, e
extraem os números de volta do relatório que ele devolve. A lógica interna é copiada
literalmente do notebook original (é a parte mais delicada do pipeline — qualquer mudança de
comportamento aqui muda os números reportados), com comentários adicionados para explicar
cada bloco.


In [ ]:
def _extract_metrics_from_report(report, metric_list):
    """Puxa os valores numéricos das métricas de dentro do objeto de relatório do PyEvALL."""
    results = {}
    if not hasattr(report, 'report') or 'metrics' not in report.report:
        return results

    metrics_dict = report.report['metrics']
    for metric in metric_list:
        if metric not in metrics_dict:
            continue
        if metrics_dict[metric].get('status') != 'OK':
            continue
        try:
            results[metric] = float(
                metrics_dict[metric]['results']['average_per_test_case']
            )
        except (KeyError, TypeError, ValueError):
            pass
    return results

In [ ]:
def _format_for_pyevall(probs, sample_ids, task, evaluation, thresholds_c=None):
    """Converte um array de probabilidades para o formato JSON que o PyEvALL espera.

    thresholds_c: lista opcional de 6 floats, usada só na Task C modo 'hard' — o valor
                  de corte por categoria para decidir se ela "está presente" ou não.
                  Se None, usa 0.5 para todas as categorias.
    """
    out = []
    for i, sid in enumerate(sample_ids):
        entry = {"test_case": "EXIST2025", "id": str(sid)}

        if task == '2_1':
            # Task A: sigmoid(logit) = P(NO) -- ver nota na seção 11 sobre a convenção
            # de índice de LABELS_2_1 = ['YES', 'NO']. probs[i][0] já vem como P(YES).
            p_yes = float(probs[i][0])
            p_no  = float(probs[i][1])
            if evaluation == 'hard':
                entry["value"] = "YES" if p_yes >= p_no else "NO"
            else:
                entry["value"] = {"YES": p_yes, "NO": p_no}

        elif task == '2_2':
            label_map = {"NONE": "NO", "JUDGEMENTAL": "JUDGEMENTAL", "DIRECT": "DIRECT"}
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_2)}
            if evaluation == 'hard':
                entry["value"] = max(relevant, key=relevant.get)
            else:
                entry["value"] = relevant

        elif task == '2_3':
            label_map = {"NONE": "NO"}
            label_map.update({lbl: lbl for lbl in LABELS_2_3[1:]})
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_3)}

            if evaluation == 'hard':
                threshs = thresholds_c if thresholds_c is not None else [0.5] * len(LABELS_2_3)
                thresh_map = {label_map[lbl]: threshs[idx] for idx, lbl in enumerate(LABELS_2_3)}
                # marca como presente toda categoria cuja probabilidade bate o threshold dela
                hard_labels = [lbl for lbl, p in relevant.items() if p >= thresh_map[lbl]]
                if not hard_labels:
                    # nenhuma categoria bateu o threshold -- em vez de devolver uma lista
                    # vazia (o PyEvALL rejeitaria), usa a categoria mais provável mesmo assim
                    hard_labels = [max(relevant, key=relevant.get)]
                entry["value"] = hard_labels
            else:
                entry["value"] = relevant

        out.append(entry)
    return out

In [ ]:
def _run_pyevall(probs, gold_dist, sample_ids, task, evaluation,
                 temp_dir='./pyevall_temp_train', thresholds_c=None):
    """Roda o PyEvALL uma vez, para uma task e um modo de avaliação (soft ou hard)."""
    os.makedirs(temp_dir, exist_ok=True)

    preds = _format_for_pyevall(probs, sample_ids, task, evaluation,
                                 thresholds_c=thresholds_c)
    # thresholds_c NÃO é passado para o gold -- o gold usa a distribuição real de votos,
    # e o PyEvALL binariza essa distribuição internamente quando o modo é 'hard'.
    gold  = _format_for_pyevall(gold_dist, sample_ids, task, evaluation)

    pred_path = f'{temp_dir}/pred_{task}_{evaluation}.json'
    gold_path = f'{temp_dir}/gold_{task}_{evaluation}.json'
    with open(pred_path, 'w') as f: json.dump(preds, f)
    with open(gold_path, 'w') as f: json.dump(gold, f)

    if evaluation == 'hard':
        metric_list = ['ICM', 'ICMNorm', 'FMeasure']
    else:
        if task == '2_3':
            metric_list = ['ICMSoft', 'ICMSoftNorm']
        else:
            metric_list = ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']

    params = {}
    if task == '2_2':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_2_HIERARCHY
    elif task == '2_3':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_3_HIERARCHY

    try:
        evaluator = PyEvALLEvaluation()
        report    = evaluator.evaluate(pred_path, gold_path, metric_list, **params)
        return _extract_metrics_from_report(report, metric_list)
    except Exception as e:
        print(f"  ! PyEvALL falhou para task {task} ({evaluation}): {e}")
        return {}

### 14.2 Busca do melhor threshold da Task C, em português claro

A Task C prevê 6 categorias independentes, cada uma com uma probabilidade entre 0 e 1. Para
virar uma decisão "está presente / não está" (modo *hard*), é preciso um valor de corte —
o **threshold**. Usar 0.5 para todas é a opção óbvia, mas nem sempre é a melhor: uma
categoria rara pode se beneficiar de um corte mais baixo (ex.: 0.30), outra mais comum de um
corte mais alto.

**Como a busca funciona, passo a passo** (chamada de "busca gulosa por coordenada"):
1. Começa com threshold 0.5 para as 6 categorias.
2. Pega a categoria 1 (`NONE`). Testa 12 valores de corte diferentes para ela (0.05, 0.10,
   ..., 0.60), **mantendo as outras 5 categorias fixas em 0.5**. Fica com o valor que deu o
   melhor ICM-Norm hard.
3. Repete o mesmo processo para a categoria 2, depois a 3, e assim por diante — cada categoria
   "herda" o melhor valor já encontrado para as anteriores.
4. No final, tem um threshold ajustado para cada uma das 6 categorias.

**Custo:** 12 valores testados × 6 categorias = 72 chamadas ao PyEvALL, **a cada época de
validação** (rodou 30 vezes durante o treino inteiro). Esse ajuste, quando aplicado, melhorou
o ICM hard da Task C de 0,3761 para 0,3948 (ver `CLAUDE.md` do repositório).


In [ ]:
def _search_thresholds_task_c(probs_c, targets_c, ids_c, temp_dir):
    """Busca gulosa por coordenada: ajusta 1 threshold por vez, mantendo os outros fixos."""
    best_thresholds = [0.5] * len(LABELS_2_3)

    for class_idx in range(len(LABELS_2_3)):
        best_t   = best_thresholds[class_idx]
        best_icm = -np.inf

        for t in THRESHOLD_GRID:
            candidate = best_thresholds.copy()
            candidate[class_idx] = float(t)

            scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                  temp_dir=temp_dir, thresholds_c=candidate)
            icm = scores.get('ICMNorm', -np.inf)
            if icm > best_icm:
                best_icm = icm
                best_t   = float(t)

        best_thresholds[class_idx] = best_t

    # avaliação final com todos os thresholds já travados nos melhores valores encontrados
    final_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                temp_dir=temp_dir, thresholds_c=best_thresholds)
    return best_thresholds, final_scores

### 14.3 `compute_metrics` — a função principal chamada pelo `Trainer`

Roda 6 avaliações (3 tasks × {soft, hard}) mais a busca de threshold da Task C, a cada época.
Convenção importante para ler o bloco da Task A: o modelo foi treinado com a loss calculada
contra `labels_a[:, 1]`, que é **P(NO)** (`LABELS_2_1 = ['YES', 'NO']`, índice 1 = NO) —
então `sigmoid(logit)` dá `P(NO)` diretamente, e `P(YES) = 1 - P(NO)`. É por isso que
`prob_no` (não `prob_yes`) é a variável calculada primeiro no bloco abaixo.


In [ ]:
def _metrics_task_a(logits_a, labels_a, temp_dir):
    """Task A: sexista sim/não. Devolve {} se nenhuma amostra tem voto válido."""
    metrics = {}
    if labels_a.ndim == 2:
        labels_a = labels_a[:, 1]

    mask_a = labels_a != -100.0
    if mask_a.any():
        prob_no  = 1 / (1 + np.exp(-logits_a[mask_a].squeeze(-1)))  # sigmoid = P(NO)
        probs_a  = np.stack([1 - prob_no, prob_no], axis=-1)        # [P(YES), P(NO)]
        targets_a = np.stack([1 - labels_a[mask_a], labels_a[mask_a]], axis=-1)
        ids_a    = [str(i) for i in range(len(probs_a))]

        hard_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'hard',
                                   temp_dir=temp_dir)
        for k, v in hard_scores.items():
            metrics[f'task_a_{k.lower()}_hard'] = v

        soft_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'soft',
                                   temp_dir=temp_dir)
        for k, v in soft_scores.items():
            metrics[f'task_a_{k.lower()}_soft'] = v

    return metrics


def _metrics_task_b(logits_b, labels_b, temp_dir):
    """Task B: intenção (NONE/JUDGEMENTAL/DIRECT). Devolve {} se nenhuma amostra tem voto válido."""
    metrics = {}
    mask_b = labels_b[:, 0] != -1.0
    if mask_b.any():
        x        = logits_b[mask_b]
        x        = x - x.max(axis=-1, keepdims=True)  # estabilidade numérica do softmax
        probs_b  = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
        targets_b = labels_b[mask_b]
        ids_b    = [str(i) for i in range(len(probs_b))]

        hard_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'hard',
                                   temp_dir=temp_dir)
        for k, v in hard_scores.items():
            metrics[f'task_b_{k.lower()}_hard'] = v

        soft_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'soft',
                                   temp_dir=temp_dir)
        for k, v in soft_scores.items():
            metrics[f'task_b_{k.lower()}_soft'] = v

    return metrics


def _metrics_task_c(logits_c, labels_c, temp_dir):
    """Task C: categorias finas (multilabel), incluindo a busca de threshold (seção 14.2).
    Devolve {} se nenhuma amostra tem voto válido.

    IMPORTANTE: a ordem das atribuições abaixo é proposital -- 'task_c_..._hard' é
    escrito primeiro pelo bloco 'hard_default' (threshold=0.5) e depois SOBRESCRITO
    pelo bloco de threshold ajustado, que se torna o score hard oficial. Não separar
    essas duas atribuições em funções diferentes chamadas fora de ordem.
    """
    metrics = {}
    mask_c = labels_c[:, 0] != -1.0
    if mask_c.any():
        probs_c  = 1 / (1 + np.exp(-logits_c[mask_c]))
        targets_c = labels_c[mask_c]
        ids_c    = [str(i) for i in range(len(probs_c))]

        soft_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'soft',
                                   temp_dir=temp_dir)
        for k, v in soft_scores.items():
            metrics[f'task_c_{k.lower()}_soft'] = v

        # threshold padrão (0.5), só de referência para comparação
        hard_scores_default = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                           temp_dir=temp_dir)
        for k, v in hard_scores_default.items():
            metrics[f'task_c_{k.lower()}_hard_default'] = v

        # threshold ajustado pela busca gulosa (seção 14.2) -- vira o score hard oficial
        best_thresholds, hard_scores_tuned = _search_thresholds_task_c(
            probs_c, targets_c, ids_c, temp_dir
        )
        for k, v in hard_scores_tuned.items():
            metrics[f'task_c_{k.lower()}_hard'] = v

        # loga os thresholds por categoria, para acompanhar como evoluem entre épocas
        for j, label in enumerate(LABELS_2_3):
            metrics[f'task_c_thresh_{label.lower().replace("-", "_")}'] = round(best_thresholds[j], 2)

    return metrics


def compute_metrics(eval_pred):
    logits_a, logits_b, logits_c = eval_pred.predictions
    labels_a, labels_b, labels_c = eval_pred.label_ids
    logits_a = np.asarray(logits_a)
    logits_b = np.asarray(logits_b)
    logits_c = np.asarray(logits_c)
    labels_a = np.asarray(labels_a)
    labels_b = np.asarray(labels_b)
    labels_c = np.asarray(labels_c)
    metrics = {}

    with tempfile.TemporaryDirectory() as temp_dir:
        metrics.update(_metrics_task_a(logits_a, labels_a, temp_dir))
        metrics.update(_metrics_task_b(logits_b, labels_b, temp_dir))
        metrics.update(_metrics_task_c(logits_c, labels_c, temp_dir))

    return metrics

## 15. Treino

Instancia o `Trainer` do HuggingFace com o modelo, os argumentos de treino, os datasets e a
função de métricas definidos acima, e roda `trainer.train()`.


### 15.1 Salvar métricas de cada época (CSV + JSON)

O `Trainer` já acumula um histórico de tudo que loga (`trainer.state.log_history`) — uma
lista de dicionários, um por evento de log/avaliação, com chaves diferentes dependendo do
tipo (passo de treino vs. avaliação de época). O callback abaixo salva esse histórico em
CSV e JSON **a cada época**, sobrescrevendo o mesmo par de arquivos — assim, se o treino for
interrompido na VM, o histórico até a última época concluída não se perde.

O nome do arquivo usa o **timestamp de início da run** (`YYYYMMDD_HHMMSS`), gerado uma vez
quando o callback é criado, para que todas as épocas de uma mesma run gravem no mesmo par de
arquivos, e runs diferentes nunca se sobrescrevam.


In [ ]:
class MetricsHistoryLogger(TrainerCallback):
    """Salva trainer.state.log_history em CSV e JSON a cada época, no arquivo
    <output_dir>/<timestamp_da_run>.{json,csv}."""

    def __init__(self, output_dir="./metrics_history"):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self.run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.json_path = os.path.join(self.output_dir, f"{self.run_timestamp}.json")
        self.csv_path = os.path.join(self.output_dir, f"{self.run_timestamp}.csv")

    def _save(self, log_history):
        with open(self.json_path, "w", encoding="utf-8") as f:
            json.dump(log_history, f, indent=2, ensure_ascii=False)

        # linhas de log têm chaves diferentes entre si (passo de treino vs. avaliação),
        # então a união de todas as chaves vira o cabeçalho do CSV; células ausentes
        # numa linha específica ficam vazias
        all_keys = []
        for row in log_history:
            for key in row:
                if key not in all_keys:
                    all_keys.append(key)

        with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=all_keys)
            writer.writeheader()
            writer.writerows(log_history)

    def on_evaluate(self, args, state, control, **kwargs):
        self._save(state.log_history)

    def on_train_end(self, args, state, control, **kwargs):
        self._save(state.log_history)


metrics_logger = MetricsHistoryLogger()
print(f"Métricas desta run serão salvas em: {metrics_logger.json_path} e {metrics_logger.csv_path}")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[metrics_logger],
)

In [ ]:
trainer.train()

## 16. Salvar o modelo treinado

`Trainer.save_model()` tem um problema conhecido quando combinado com um `nn.Module`
customizado (como o nosso `MultiHeadModel`) + `Accelerate` + `fp16`: o state_dict salvo pode
referenciar pesos iniciais desatualizados em vez dos pesos finais treinados. Para evitar isso,
extraímos o modelo real de dentro do wrapper de paralelismo do Accelerate e salvamos os pesos
manualmente via `safetensors`.


In [ ]:
import safetensors.torch
from accelerate.utils import extract_model_from_parallel

actual_model = extract_model_from_parallel(trainer.model)
state_dict = {k: v.cpu().clone() for k, v in actual_model.state_dict().items()}
safetensors.torch.save_file(state_dict, "./best_model/model.safetensors")
tokenizer.save_pretrained("./best_model")
print("Modelo e tokenizer salvos em ./best_model")
print("Fingerprint dos pesos finais:", model_fingerprint(actual_model))

## 17. Diagnósticos: o modelo está realmente usando a imagem?

Depois do treino, vale checar se cada modalidade está mesmo contribuindo, ou se o modelo
"aprendeu a ignorar" alguma delas (o que pode acontecer se um sinal for muito mais fácil de
usar que o outro). Duas checagens simples:


### 17.1 A visão influencia a previsão?

Roda a validação **duas vezes** com o mesmo modelo: uma com as imagens de verdade, outra
com as imagens trocadas por zeros (equivalente a "imagem em branco"). Depois compara as
previsões da Task A nos dois casos:

- Se as previsões forem **quase idênticas** (correlação perto de 1.0), a imagem não estava
  influenciando nada — o modelo estava decidindo só pelo texto.
- Se forem **bem diferentes** (correlação baixa), a imagem está de fato mudando a decisão do
  modelo.


In [ ]:
def diagnose_vision_contribution(model, val_loader, device='cuda'):
    model.eval()

    results = {'real_images': [], 'zero_images': []}

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            real_pixels = batch['pixel_values'].to(device)
            zero_pixels = torch.zeros_like(real_pixels)

            _, logits_a_real, logits_b_real, logits_c_real = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=real_pixels,
            )

            _, logits_a_zero, logits_b_zero, logits_c_zero = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=zero_pixels,
            )

            probs_a_real = torch.sigmoid(logits_a_real).cpu().numpy()
            probs_a_zero = torch.sigmoid(logits_a_zero).cpu().numpy()

            results['real_images'].append(probs_a_real)
            results['zero_images'].append(probs_a_zero)

    real = np.concatenate(results['real_images']).flatten()
    zero = np.concatenate(results['zero_images']).flatten()

    diff = np.abs(real - zero)
    correlation = np.corrcoef(real, zero)[0, 1]

    print("=" * 60)
    print("DIAGNÓSTICO DE ABLAÇÃO DE VISÃO")
    print("=" * 60)
    print(f"Diferença média absoluta entre previsões com imagem real vs zerada:")
    print(f"  {diff.mean():.4f}  (apenas Task A)")
    print(f"")
    print(f"Correlação entre previsões reais e zeradas: {correlation:.4f}")
    print(f"  - 1.00 = a imagem não tem NENHUM efeito (modelo ignora imagens)")
    print(f"  - 0.50 = contribuição moderada da imagem")
    print(f"  - 0.00 = imagem domina a decisão / aleatório")
    print(f"")

    if correlation > 0.99:
        print("[AVISO] A imagem está sendo essencialmente ignorada!")
    elif correlation > 0.95:
        print("[AVISO] A imagem contribui muito pouco.")
    elif correlation > 0.85:
        print("[AVISO] A imagem contribui de forma modesta.")
    else:
        print("[OK] A imagem está contribuindo de forma significativa.")

    return correlation, diff.mean()

In [ ]:
diagnose_vision_contribution(actual_model, val_loader)

### 17.2 O CLIP está vendo imagens diferentes, ou tudo igual?

Antes de confiar no diagnóstico acima, vale confirmar que o CLIP está mesmo recebendo imagens
diferentes entre si — se por algum motivo todas as imagens caíssem no fallback preto (seção
10), a checagem anterior daria "correlação alta" por um motivo errado (não porque o modelo
ignora a imagem, mas porque todas as imagens são iguais).

Esta função pega 20 amostras, roda só o codificador de imagem (CLIP), e mede a **similaridade
de cosseno** entre cada par de saídas — quão "parecidos" os vetores são entre si, numa escala
de -1 (opostos) a 1 (idênticos). Se todas as saídas forem quase idênticas (~1.0), tem sinal
de que o pré-processamento de imagem está quebrado.


In [ ]:
def check_clip_output_diversity(model, val_loader, device='cuda', n_samples=20):
    model.eval()
    cls_outputs = []

    with torch.no_grad():
        count = 0
        for batch in val_loader:
            pixel_values = batch['pixel_values'].to(device)

            vision_out = model.vision_encoder(pixel_values=pixel_values)
            cls_token = vision_out.last_hidden_state[:, 0, :]  # [batch, 1024]

            cls_outputs.append(cls_token.cpu().numpy())

            count += pixel_values.shape[0]
            if count >= n_samples:
                break

    cls_outputs = np.concatenate(cls_outputs, axis=0)[:n_samples]

    from numpy.linalg import norm
    similarities = []
    for i in range(len(cls_outputs)):
        for j in range(i + 1, len(cls_outputs)):
            sim = np.dot(cls_outputs[i], cls_outputs[j]) / (
                norm(cls_outputs[i]) * norm(cls_outputs[j]) + 1e-8
            )
            similarities.append(sim)

    similarities = np.array(similarities)

    print("=" * 60)
    print("CHECAGEM DE DIVERSIDADE DE SAÍDA DO CLIP")
    print("=" * 60)
    print(f"Similaridade de cosseno entre pares, {n_samples} amostras:")
    print(f"  Média: {similarities.mean():.4f}")
    print(f"  Desvio padrão:  {similarities.std():.4f}")
    print(f"  Mínimo:  {similarities.min():.4f}")
    print(f"  Máximo:  {similarities.max():.4f}")
    print()

    if similarities.mean() > 0.99:
        print("[AVISO] As saídas do CLIP são quase idênticas para todas as imagens!")
        print("        Isso indica que o CLIP não está vendo conteúdo diverso.")
        print("        Causas possíveis: todas as imagens caíram no fallback preto,")
        print("                          ou o pré-processamento está quebrado.")
    elif similarities.mean() > 0.95:
        print("[AVISO] As saídas do CLIP são muito parecidas (baixa diversidade).")
    else:
        print("[OK] O CLIP está produzindo saídas diversas.")

    return similarities

check_clip_output_diversity(actual_model, val_loader)

## 18. Próximos passos

**Este notebook NÃO reproduz mais o sistema submetido.** Duas correções foram aplicadas
(2026-08-01) sobre a base originalmente fiel ao `Thesis_main.ipynb`:

1. **Segmentação de hashtags corrigida** (seção 6) — `wordsegment.load()` agora é chamado.
   Muda o texto de entrada em 10/3.984 amostras.
2. **Apóstrofo do vocabulário de escolaridade corrigido** (seção 8) — `_normalize_apostrophes`
   trata `’`/`'` como equivalentes. Muda o vetor demográfico em 3.957/3.984 amostras.

Isso significa que **qualquer número reportado a partir de um treino deste notebook não é
comparável diretamente** aos números do sistema submetido (ICM hard 0,3761→0,3948 etc., ver
`CLAUDE.md`) — este é um **baseline novo**, e precisa de um treino do zero para gerar números
próprios antes de qualquer comparação ou estudo de ablação.

O que foi efetivamente verificado nesta revisão:

- Pré-processamento de texto, remapeamento de labels, soft labels e vetor demográfico:
  comparados função a função contra o notebook original em **todas as 3.984 amostras** do
  dataset de treino, incluindo comparação em processos isolados antes/depois dos dois fixes
  acima (para garantir que cada fix muda exatamente o que deveria, nem mais nem menos).
- Lógica do modelo (`MultiHeadModel.forward`, incluindo as flags novas `use_vision` /
  `use_demographics`, e a extração dos métodos `_loss_task_a/b/c` para reduzir complexidade
  cognitiva): verificada numericamente contra a versão anterior em 5 cenários (com/sem
  imagem, com/sem demografia, labels mascaradas parcial e totalmente) — perda e logits
  idênticos, testado direto na classe `MultiHeadModel` extraída do notebook (não uma cópia).
- `compute_metrics` (dividida em `_metrics_task_a/b/c` para reduzir complexidade cognitiva):
  cada bloco comparado isoladamente contra o original — únicas diferenças são a estrutura de
  função (def/docstring/`return`), nenhuma linha de lógica mudou.
- Warnings do SonarQube endereçados: closure sobre variável de loop (célula 3.1), dict
  comprehension → `dict.fromkeys` (seções 7), complexidade cognitiva de
  `compute_annotator_vector` (26→ok), `forward` (20→ok) e `compute_metrics` (→ok).

**O que NÃO foi verificado:** este notebook não foi executado de ponta a ponta. Não há GPU
disponível localmente, e XLM-RoBERTa-large + CLIP ViT-L/14 nunca foram baixados aqui. A
primeira execução real deve acontecer na VM de treino — o ponto de falha mais provável é
algum import faltando na célula 5 (ela foi reconstruída a partir do código, não copiada
literalmente da célula de imports do notebook original).

As flags `use_vision` e `use_demographics` no construtor de `MultiHeadModel` (seção 11) —
ligadas por padrão em `True`/`True` — preparam o terreno para o próximo passo combinado:
**o estudo de ablação**, testando o modelo com e sem cada modalidade para quantificar a
contribuição real de cada uma, agora a partir deste baseline corrigido.